# [P2] Q1 B + D: analysis and realizations

**Owner:** Part 2  
**Temporary part notebook.** At integration it is merged into `Q1_neural_spikes.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q1b` (analysis and comparison), `t_q1d` (new realizations, how to tell the patterns apart).

Decisions to read on the team board first (board ids): `b_*` (B), `d_n d_props d_plot d_cls d_dur` (D), and the shared `a_rate`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


## Q1 B, step 1: the statistics functions

Decisions applied: `b_stats` (rate and mean interval, CV, Fano, LV, short-interval fraction, serial correlation, CV2), `b_thr` (a curve over thresholds), `b_fano` (a curve over window sizes) and `b_bin` (100 ms bins).

Each function takes the spike times in seconds and returns a number, or `np.nan` when the number is undefined (too few spikes, division by zero). Nothing here depends on the simulation yet: the functions are tested on inputs whose answer we know in advance (decision `g_verify`).

In [ ]:
import numpy as np
from q1_stats import (isi, mean_rate, mean_isi, cv, lv, cv2, isi_lag1_corr, burst_fraction, burst_curve,
                      window_counts, fano, fano_curve, rate_over_time)

# HE: בדיקה 1: מקרים שאת התשובה שלהם אפשר לחשב ביד.
# HE: פעימות ב-0, 1, 4, 5 שניות => מרווחים 1, 3, 1.
s = np.array([0.0, 1.0, 4.0, 5.0])
assert np.allclose(isi(s), [1, 3, 1])
assert mean_rate(s, T=10) == 0.4                      # HE: 4 פעימות ב-10 שניות
assert np.isclose(mean_isi(s), 5 / 3)

# HE: שני מרווחים בלבד, 1 ו-3: ממוצע 2, סטיית תקן (ddof=1) היא שורש 2, ולכן CV = שורש 2 חלקי 2.
two = np.array([0.0, 1.0, 4.0])
assert np.isclose(cv(two), np.sqrt(2) / 2)
# HE: LV = 3 * ((1-3)/(1+3))^2 = 3 * 0.25 = 0.75, ו-CV2 = 2*|1-3|/(1+3) = 1.
assert np.isclose(lv(two), 0.75)
assert np.isclose(cv2(two), 1.0)

# HE: מרווחים מתחלפים 1,2,1,2,1: כל קצר בא אחרי ארוך, ולכן המתאם הסדרתי הוא בדיוק -1.
alt = np.concatenate([[0.0], np.cumsum([1, 2, 1, 2, 1])])
assert np.isclose(isi_lag1_corr(alt), -1.0)
print("hand-computed cases passed")

In [ ]:
# HE: בדיקה 2: נוירון רגולרי לגמרי (פעימה כל 100 מילי-שניות) הוא "אפס שונות": CV, LV, CV2 ו-Fano שווים 0.
regular = np.arange(0, 10, 0.1)
assert np.isclose(cv(regular), 0, atol=1e-9) and np.isclose(lv(regular), 0, atol=1e-9)
assert np.isclose(cv2(regular), 0, atol=1e-9)
assert np.isclose(fano(regular, T=10, window=0.5), 0, atol=1e-9)   # HE: בכל חלון בדיוק 5 פעימות
assert np.isnan(isi_lag1_corr(regular))                             # HE: אין שונות, אין מתאם: nan
assert burst_fraction(regular, 0.010) == 0                          # HE: אין מרווח קצר מ-10 מילי-שניות

# HE: בדיקה 3: נוירון אקראי (פואסוני) צריך לתת בערך 1 בשלושת המדדים CV, LV ו-Fano. מחוללים אותו כאן עם זרע קבוע.
rng = np.random.default_rng(1)
poisson = np.cumsum(rng.exponential(1 / 10, size=20000))            # HE: קצב 10Hz, כ-2000 שניות
T_long = poisson[-1]
poisson = poisson[:-1]
for name, value in [("CV", cv(poisson)), ("LV", lv(poisson)),
                    ("Fano(500ms)", fano(poisson, T_long, 0.5)), ("CV2", cv2(poisson))]:
    print(f"{name:12s}", round(value, 3))
    assert abs(value - 1) < 0.1 if name != "CV2" else abs(value - 1) < 0.15, name
# HE: במרווחים אקראיים אין זיכרון, לכן המתאם הסדרתי קרוב ל-0.
assert abs(isi_lag1_corr(poisson)) < 0.05
# HE: במרווחים מעריכיים, שיעור המרווחים הקצרים מ-10ms הוא 1-exp(-10Hz*0.01s) = 0.0952.
print("short-interval fraction:", round(burst_fraction(poisson, 0.010), 4), "expected", round(1 - np.exp(-0.1), 4))
assert abs(burst_fraction(poisson, 0.010) - (1 - np.exp(-0.1))) < 0.01
print("regular and random-neuron cases passed")

In [ ]:
# HE: בדיקה 4: מקרי קצה. כל מדד שאי אפשר לחשב מחזיר nan, ולא שגיאה (כלל מההוראות).
empty = np.array([])
one = np.array([0.5])
three = np.array([0.1, 0.2, 0.35])
for f in (mean_isi, cv, lv, cv2, isi_lag1_corr, burst_fraction):
    assert np.isnan(f(empty)) and np.isnan(f(one)), f.__name__
assert np.isnan(cv(one)) and np.isnan(lv(three[:2])) and not np.isnan(lv(three))
assert np.isnan(fano(empty, T=5, window=0.5))                       # HE: ממוצע ספירות 0 => nan
assert np.isnan(fano(three, T=0.4, window=0.5))                     # HE: פחות משני חלונות => nan
assert mean_rate(empty, T=5) == 0

# HE: חלונות: הקלטה של 5 שניות וחלון של 0.5 נותנים בדיוק 10 חלונות, וסכום הספירות שווה למספר הפעימות.
c = window_counts(poisson[poisson < 5], T=5, window=0.5)
assert len(c) == 10 and c.sum() == int((poisson < 5).sum())
# HE: קצב לאורך זמן בחלונות של 100ms: 50 חלונות, וממוצע הקצב שווה לקצב הכולל.
r = rate_over_time(poisson[poisson < 5], T=5, bin_size=0.1)
assert len(r) == 50 and np.isclose(r.mean(), len(poisson[poisson < 5]) / 5)
# HE: העקומות מחזירות ערך לכל סף/חלון, באורך שווה לרשימה שהוזנה.
assert len(burst_curve(poisson, np.linspace(0.002, 0.03, 8))) == 8
assert len(fano_curve(poisson, T_long, [0.05, 0.1, 0.5])) == 3
print("edge cases passed: undefined values return nan, never an exception")

**What step 1 shows**

- The functions agree with answers computed by hand (for example, intervals 1 and 3 give CV = 0.707, LV = 0.75, CV2 = 1).
- A perfectly regular neuron gives CV, LV, CV2 and Fano equal to 0, and an undefined serial correlation (`nan`).
- A random (Poisson) neuron gives CV, LV, CV2 and Fano close to 1 (0.998, 1.001, 1.001, 0.996) and a serial correlation near 0. The short-interval fraction below 10 ms is 0.098, close to the theoretical 0.095.
- Undefined cases (no spikes, one spike, too few windows) return `nan`, never an exception.

## Q1 B, step 2: temporary stand-in neurons, raster and ISI histograms

**Important:** Part 1 has not delivered the simulation yet (`simulate_regular`, `simulate_irregular`, `simulate_bursting` in `q1_models.py`). Until it does, the cells below use **temporary stand-in generators** written only to build and check the figures. They are not the models of the assignment and will be replaced by Part 1's functions (the cell is marked `STAND-IN`). All three neurons have the same mean rate of 10 Hz (decision `a_rate`).

In [ ]:
# STAND-IN (temporary): replace with q1_models.simulate_* when Part 1 delivers them.
# HE: מחוללים זמניים כדי לבנות גרפים. לא המודלים האמיתיים של שחר. כולם עם קצב ממוצע של 10Hz.
import numpy as np
import matplotlib.pyplot as plt

RATE = 10.0            # HE: קצב ממוצע בהרץ (החלטה a_rate)
NAMES = ["Regular", "Irregular (random)", "Bursting"]
COLORS = ["tab:blue", "tab:orange", "tab:green"]


def _to_spike_times(intervals, T):
    # HE: צוברים מרווחים לזמני פעימות, וגוזרים כך שהכול בתוך [0, T).
    t = np.cumsum(intervals)
    return t[t < T]


def standin_regular(T, rng):
    # HE: מרווחים סביב 100ms עם סטייה קטנה (התפלגות גמא עם צורה גדולה = כמעט קבוע).
    k = 25
    return _to_spike_times(rng.gamma(k, 1 / (RATE * k), size=int(RATE * T * 2) + 20), T)


def standin_irregular(T, rng):
    # HE: תהליך פואסוני: מרווחים מעריכיים.
    return _to_spike_times(rng.exponential(1 / RATE, size=int(RATE * T * 2) + 20), T)


def standin_bursting(T, rng, per_burst=5, within=0.005):
    # HE: פרצים: כל פרץ הוא 5 פעימות במרווחים של כ-5ms, ופרצים מתחילים באקראי בקצב RATE/5.
    starts = _to_spike_times(rng.exponential(per_burst / RATE, size=int(RATE * T) + 20), T)
    spikes = np.concatenate([s + np.concatenate([[0], np.cumsum(rng.exponential(within, per_burst - 1))])
                             for s in starts])
    spikes = np.sort(spikes[spikes < T])
    return spikes


def make_standins(T, seed):
    rng = np.random.default_rng(seed)
    return [standin_regular(T, rng), standin_irregular(T, rng), standin_bursting(T, rng)]


# HE: ריצה קצרה (5 שניות) לרסטר, וריצה ארוכה (200 שניות) כדי שההיסטוגרמות יהיו יציבות.
T_SHORT, T_LONG = 5.0, 200.0
short = make_standins(T_SHORT, seed=11)
long_ = make_standins(T_LONG, seed=12)

# HE: בדיקה: הקצב הממוצע של שלושת המחוללים קרוב ל-10Hz, והזמנים עולים בקפדנות ובתוך ההקלטה.
for name, s, T in zip(NAMES, long_, [T_LONG] * 3):
    assert np.all(np.diff(s) > 0) and s[0] >= 0 and s[-1] < T, name
    print(f"{name:20s} mean rate {len(s) / T:5.2f} Hz")
    assert abs(len(s) / T - RATE) < 1.0, name

In [ ]:
# HE: גרף 1: רסטר (החלטה b_figs). כל פעימה היא קו אנכי קטן, שורה לכל נוירון, ציר x הוא הזמן.
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.eventplot(short, colors=COLORS, lineoffsets=[2, 1, 0], linelengths=0.7)
ax.set_yticks([2, 1, 0]); ax.set_yticklabels(NAMES)
ax.set_xlim(0, T_SHORT); ax.set_xlabel("Time (s)")
ax.set_title("Raster: same mean rate (10 Hz), different timing structure")
plt.tight_layout(); plt.show()

In [ ]:
# HE: גרף 2: היסטוגרמת מרווחים (ISI) בציר לוגריתמי (החלטה b_axis). ב-bins בלוגריתם, רוחב הפח גדל עם הערך.
from q1_stats import isi
bins = np.logspace(-3, 0.5, 40)            # HE: מ-1ms עד כ-3 שניות
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharex=True, sharey=True)
for ax, name, color, s in zip(axes, NAMES, COLORS, long_):
    ax.hist(isi(s), bins=bins, color=color)
    ax.set_xscale("log"); ax.set_title(name); ax.set_xlabel("Interval (s, log axis)")
axes[0].set_ylabel("Count")
plt.tight_layout(); plt.show()

**What step 2 shows (on stand-in data)**

- **Raster:** all three rows have about 10 spikes per second, yet they look different: the regular neuron is evenly spaced, the random one has clusters and gaps by chance, and the bursting one fires tight groups separated by long silences. Mean rate alone does not tell them apart.
- **ISI histograms (log axis):** the regular neuron has one narrow peak near 100 ms; the random neuron one wide hump; the bursting neuron two humps, short intervals inside bursts (about 5 ms) and long ones between bursts (hundreds of ms). On a linear axis the short bump would be squeezed against zero.
- All of this is for stand-in data. The same cells will be re-run on Part 1's simulation.

## Q1 B, step 3: rate over time, return map, threshold curve and Fano curve

Decisions applied: `b_bin` (100 ms bins), `b_figs` (rate over time, return map), `b_thr` (a curve over thresholds from 2 to 30 ms) and `b_fano` (a curve over window sizes). Still on the stand-in neurons of step 2.

In [ ]:
from q1_stats import rate_over_time, burst_curve, fano_curve

# HE: גרף 3: קצב לאורך זמן בחלונות של 100ms (החלטה b_bin). חלון קצר מראה פרצים אבל רועש: זה בדיוק הלקח.
fig, ax = plt.subplots(figsize=(10, 3))
t_bins = np.arange(int(T_SHORT / 0.1)) * 0.1 + 0.05
for name, color, s in zip(NAMES, COLORS, short):
    ax.plot(t_bins, rate_over_time(s, T_SHORT, 0.100), color=color, label=name, lw=1.5)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Rate (Hz, 100 ms bins)"); ax.legend(ncol=3, loc="upper right")
ax.set_title("Rate in 100 ms bins: the same mean, very different fluctuations")
plt.tight_layout(); plt.show()

In [ ]:
# HE: גרף 4: מפת חזרה. כל נקודה היא מרווח (ציר x) מול המרווח שאחריו (ציר y), בצירים לוגריתמיים.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for ax, name, color, s in zip(axes, NAMES, COLORS, long_):
    x = isi(s)
    ax.scatter(x[:-1], x[1:], s=3, alpha=0.3, color=color)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(name); ax.set_xlabel("Interval n (s)")
    # HE: מגבילים את הצירים לטווח הרלוונטי (1ms עד 3 שניות), כדי שחריגים נדירים לא ימתחו את הגרף.
    ax.set_xlim(1e-3, 3); ax.set_ylim(1e-3, 3)
axes[0].set_ylabel("Interval n+1 (s)")
plt.tight_layout(); plt.show()

In [ ]:
# HE: גרף 5: עקומת סף (החלטה b_thr): שיעור המרווחים הקצרים מהסף, לכל סף בין 2 ל-30 מילי-שניות.
thresholds = np.linspace(0.002, 0.030, 15)
fig, ax = plt.subplots(figsize=(6, 3.4))
for name, color, s in zip(NAMES, COLORS, long_):
    ax.plot(thresholds * 1000, burst_curve(s, thresholds), color=color, marker="o", ms=3, label=name)
ax.set_xlabel("Short-interval threshold (ms)"); ax.set_ylabel("Fraction of intervals below threshold")
ax.legend(); ax.set_title("How the answer depends on the threshold")
plt.tight_layout(); plt.show()

# HE: בדיקות סבירות על הנתונים: ברגולרי אין מרווחים קצרים מ-10ms, בפרצי רוב המרווחים קצרים, באקראי כ-1-exp(-0.1)=0.095.
at10 = {n: burst_curve(s, [0.010])[0] for n, s in zip(NAMES, long_)}
print({k: round(v, 3) for k, v in at10.items()})
assert at10["Regular"] == 0 and at10["Bursting"] > 0.5 and abs(at10["Irregular (random)"] - 0.095) < 0.02

In [ ]:
# HE: גרף 6: עקומת Fano (החלטה b_fano): Fano לכל גודל חלון, בציר x לוגריתמי. הקו האפור ב-1 הוא נוירון אקראי.
windows = np.logspace(-2, 1, 16)            # HE: מ-10ms עד 10 שניות
fig, ax = plt.subplots(figsize=(6, 3.4))
curves = {}
for name, color, s in zip(NAMES, COLORS, long_):
    curves[name] = fano_curve(s, T_LONG, windows)
    ax.plot(windows, curves[name], color=color, marker="o", ms=3, label=name)
ax.axhline(1, color="gray", ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("Window size (s)"); ax.set_ylabel("Fano factor"); ax.legend()
ax.set_title("Fano factor depends on the time scale")
plt.tight_layout(); plt.show()

# HE: בדיקות סבירות: רגולרי נמוך מ-1, אקראי קרוב ל-1, ובפרצי גדל עם החלון (בחלון של שנייה כבר הרבה מעל 1).
i1 = int(np.argmin(np.abs(windows - 1.0)))
print({n: round(float(c[i1]), 2) for n, c in curves.items()}, "at window", round(windows[i1], 2), "s")
assert curves["Regular"][i1] < 0.3
assert abs(curves["Irregular (random)"][i1] - 1) < 0.25
assert curves["Bursting"][i1] > 2 and curves["Bursting"][i1] > curves["Bursting"][0]

**What step 3 shows (on stand-in data)**

- **Rate in 100 ms bins:** the regular neuron stays flat near 10 Hz, the random one fluctuates, and the bursting one jumps between 0 and about 100 Hz. A single 100 ms window is a very noisy estimate of the rate, so a short-window rate is not a reliable measure.
- **Return map:** the regular neuron is one small cloud (every interval is followed by a similar one); the random neuron is one wide cloud; the bursting one has three clouds: short followed by short (inside a burst), and short-to-long and long-to-short (the edges of a burst). The histogram cannot show this order.
- **Threshold curve:** at 10 ms the short-interval fraction is 0 for the regular neuron, 0.08 for the random one (theory 0.095) and 0.72 for the bursting one. The bursting curve flattens beyond about 10 ms, so the conclusion does not depend on the exact threshold, while the random neuron's curve keeps rising slowly.
- **Fano curve:** the regular neuron is below 1 (0.06 at 1 s), the random neuron stays near 1, and the bursting neuron is far above 1 (3.8 at 1 s) and depends on the window. The difference between neurons is a matter of time scale.